# Rogers × Databricks — Waterfront Station Cell Traffic EDA

**Goal:** understand who arrives at Waterfront Station, when, and how long they stay, as groundwork for a smart-community **transit + security** solution.

**Data:** one station (Waterfront), ~8M device records: `location_name, longitude, latitude, timestamp, origin, dwell_time`.
Heavy aggregation stays in Spark, and only small aggregates go to pandas for plotting. Runs on serverless (no `.cache()`).

| # | Section | Question |
|---|---|---|
| 0 | Setup | config, enriched Delta table, time resolution |
| 1 | Data quality | nulls, duplicates, ranges, coverage |
| 2 | Synthetic-data checks | generator artefacts, burstiness |
| 3 | Origin volume | which neighbourhoods feed the station |
| 4 | Arrival patterns over time | peaks, waves, cumulative arrivals |
| 5 | Origin × time | when each neighbourhood arrives |
| 6 | Dwell time | pass-through vs lingering, by origin & time |
| 7 | Occupancy | how many people are in the station at once |
| 8 | Security signals | surges, crowd-mix shifts, loitering |
| 9 | Relationships | does crowding change dwell? similar origins |
| 10 | Catchment map | flows from neighbourhoods into Waterfront |
| 11 | KPI tables | saved to Delta for the solution / dashboard |

## 0. Setup

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from builtins import min, max, sum, abs, round   # restores Python built-ins if `from pyspark.sql.functions import *` ran anywhere
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# ---------------- CONFIG ----------------
TABLE = "rogersdatabricks.default.synthetic_data_waterfront"
OUT_SCHEMA = "workspace.default"      # writable schema for intermediate + KPI tables
LOCAL_TZ = "UTC"                      # "UTC" = analyse clock times exactly as written; "America/Vancouver" if they are true UTC
DWELL_UNIT_SECONDS = 60               # 60 if dwell_time is minutes, 1 if seconds (confirm with organisers)
DROP_DUPLICATES = True
TOP_N = 12                            # origins shown individually; the rest become "Other"
DWELL_BINS = [0, 15, 60, 180, float("inf")]
DWELL_LABELS = ["pass-through (<15)", "short (15–60)", "medium (60–180)", "long (180+)"]
Z_THRESH = 3.5                        # robust z-score threshold for surges
CAPACITY_SCENARIOS = []               # e.g. [20000, 40000], drawn as reference lines on occupancy charts
SAMPLE_ROWS = 300_000                 # for raw-point plots
# ----------------------------------------

spark.conf.set("spark.sql.session.timeZone", LOCAL_TZ)

plt.rcParams.update({
    "figure.figsize": (13, 5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 13,
})
sns.set_palette("tab10")

def show():
    plt.tight_layout()
    plt.show()

def materialize(sdf, name):
    """Serverless-safe replacement for .cache(): write to Delta and read back."""
    full_name = f"{OUT_SCHEMA}.eda_{name}"
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)
    return spark.table(full_name)

def bucket(col, seconds):
    return F.expr(f"timestamp_seconds(floor(unix_timestamp({col}) / {seconds}) * {seconds})")

def fmt_k(x, _=None):
    return f"{x/1e6:.1f}M" if abs(x) >= 1e6 else (f"{x/1e3:.0f}K" if abs(x) >= 1e3 else f"{x:.0f}")

In [ ]:
df_raw = spark.table(TABLE)
N_RAW = df_raw.count()
N_DEDUP = df_raw.dropDuplicates().count()
src = df_raw.dropDuplicates() if DROP_DUPLICATES else df_raw

ts_expr = F.to_timestamp("timestamp") if dict(df_raw.dtypes)["timestamp"] == "string" else F.col("timestamp")

enriched = (src
    .withColumn("ts", ts_expr)
    .withColumn("dwell_time", F.col("dwell_time").cast("double"))
    .withColumn("end_ts", F.expr(f"timestamp_seconds(unix_timestamp(ts) + CAST(dwell_time * {DWELL_UNIT_SECONDS} AS BIGINT))"))
    .withColumn("date", F.to_date("ts"))
    .withColumn("hour", F.hour("ts"))
    .withColumn("minute", F.minute("ts"))
    .withColumn("second", F.second("ts"))
    .withColumn("dow", F.expr("weekday(ts)"))          # 0 = Monday
    .withColumn("dow_name", F.date_format("ts", "EEE"))
    .withColumn("is_weekend", F.col("dow") >= 5))

df = materialize(enriched, "waterfront_enriched")
N = df.count()
print(f"Raw rows: {N_RAW:,} | exact duplicates: {N_RAW - N_DEDUP:,} ({(N_RAW - N_DEDUP) / N_RAW:.2%}) | rows used: {N:,}")

In [ ]:
# Time span -> pick sensible resolutions automatically
rng = df.agg(F.min("ts").alias("first"), F.max("ts").alias("last"),
             F.max("end_ts").alias("last_end"),
             F.countDistinct("date").alias("n_days"),
             F.countDistinct("origin").alias("n_origins")).first()

SPAN_MIN = (rng["last"] - rng["first"]).total_seconds() / 60
N_DAYS = rng["n_days"]
MULTI_DAY = N_DAYS >= 3

def pick_res(span_minutes, target_bins):
    for r in [1, 2, 5, 10, 15, 30, 60, 120, 240, 1440]:
        if span_minutes / r <= target_bins:
            return r
    return 1440

RES_MIN = pick_res(SPAN_MIN, 720)                        # time-series resolution
RES_S = RES_MIN * 60
HEAT_MIN = 60 if MULTI_DAY else pick_res(SPAN_MIN, 40)   # heatmap column width

if MULTI_DAY:
    tbin_expr, TBIN_LABEL = F.lpad(F.col("hour").cast("string"), 2, "0"), "hour of day"
else:
    fmt = "HH:mm" if N_DAYS == 1 else "MM-dd HH:mm"
    tbin_expr, TBIN_LABEL = F.date_format(bucket("ts", HEAT_MIN * 60), fmt), f"{HEAT_MIN}-min window"

df = (df.withColumn("t", bucket("ts", RES_S))
        .withColumn("t_end", bucket("end_ts", RES_S))
        .withColumn("tbin", tbin_expr))

print(f"First arrival : {rng['first']}")
print(f"Last arrival  : {rng['last']}   (last departure {rng['last_end']})")
print(f"Span          : {SPAN_MIN:,.0f} min over {N_DAYS} day(s)  -> multi-day analysis: {MULTI_DAY}")
print(f"Origins       : {rng['n_origins']}")
print(f"Time-series resolution: {RES_MIN} min | heatmap columns: {TBIN_LABEL}")

## 1. Data quality

In [ ]:
df_raw.printSchema()
display(df.limit(20))

In [ ]:
base_cols = df_raw.columns
nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in base_cols]).toPandas().T
nulls.columns = ["nulls"]
nulls["null_pct"] = nulls["nulls"] / N * 100
nulls["approx_distinct"] = df.select([F.approx_count_distinct(c).alias(c) for c in base_cols]).toPandas().T[0]
display(nulls.reset_index().rename(columns={"index": "column"}))

In [ ]:
display(df.select("dwell_time").summary("count", "mean", "stddev", "min", "1%", "5%", "25%", "50%", "75%", "95%", "99%", "max"))

dq = df.agg(
    F.sum((F.col("dwell_time") <= 0).cast("int")).alias("dwell_non_positive"),
    F.sum((F.col("dwell_time") != F.floor("dwell_time")).cast("int")).alias("dwell_non_integer"),
    F.sum(F.col("ts").isNull().cast("int")).alias("unparseable_timestamps"),
).first().asDict()
print(dq)

In [ ]:
# Location / coordinate consistency (expect exactly one row)
display(df.groupBy("location_name", "latitude", "longitude").count())

# Origin labels: look for typos, casing or whitespace variants
display(df.groupBy("origin").count().orderBy("origin"))

In [ ]:
# Coverage: are there gaps in the arrival stream?
cov = df.groupBy("t").count().toPandas().set_index("t")["count"]
full_idx = pd.date_range(cov.index.min(), cov.index.max(), freq=f"{RES_MIN}min")
missing = full_idx.difference(cov.index)
print(f"{RES_MIN}-min buckets with arrivals: {len(cov):,} / {len(full_idx):,} ({len(cov) / len(full_idx):.1%})")
if len(missing):
    print("First empty buckets:", list(missing[:10]))

## 2. Synthetic-data checks
Generated data often has tell-tale patterns (only some minutes used, uniform seconds, perfectly smooth rates). Knowing them stops us from presenting generator artefacts as insights.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col in zip(axes, ["hour", "minute", "second"]):
    p = df.groupBy(col).count().orderBy(col).toPandas()
    ax.bar(p[col], p["count"], color="steelblue")
    ax.set_title(f"Records by {col}")
    ax.set_xlabel(col)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_k))
show()

In [ ]:
# Burstiness: arrivals per second vs a random (Poisson) process
per_sec = df.groupBy(F.date_trunc("second", "ts").alias("s")).count()
dist = per_sec.groupBy("count").agg(F.count("*").alias("n_seconds")).toPandas()
total_secs = int((rng["last"] - rng["first"]).total_seconds()) + 1
zero_secs = total_secs - int(dist["n_seconds"].sum())
if zero_secs > 0:
    dist = pd.concat([dist, pd.DataFrame({"count": [0], "n_seconds": [zero_secs]})])

mean = np.average(dist["count"], weights=dist["n_seconds"])
var = np.average((dist["count"] - mean) ** 2, weights=dist["n_seconds"])
print(f"Mean arrivals/second: {mean:,.2f} | variance: {var:,.2f} | index of dispersion (var/mean): {var / mean:,.2f}")
print("≈1 → random, independent arrivals · >1 → bursty (groups arriving together, e.g. train/SeaBus unloads) · <1 → unnaturally regular")

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.hist(dist["count"], weights=dist["n_seconds"], bins=60, color="slateblue")
ax.set_xlabel("arrivals in a single second")
ax.set_ylabel("number of seconds")
ax.set_title("Distribution of arrivals per second")
show()

## 3. Origin volume: which neighbourhoods feed Waterfront?

In [ ]:
org_vol = df.groupBy("origin").count().orderBy(F.desc("count")).toPandas()
org_vol["share"] = org_vol["count"] / org_vol["count"].sum()
TOP_ORIGINS = org_vol["origin"].head(TOP_N).tolist()
print(f"Distinct origins: {len(org_vol)}")

p = org_vol.head(40).iloc[::-1]
fig, ax = plt.subplots(figsize=(13, max(5, 0.35 * len(p))))
bars = ax.barh(p["origin"], p["count"], color="darkorange")
for b, s in zip(bars, p["share"]):
    ax.text(b.get_width(), b.get_y() + b.get_height() / 2, f" {s:.1%}", va="center", fontsize=9)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(fmt_k))
ax.set_title("Records by origin")
show()

In [ ]:
cum = org_vol["share"].cumsum().values
n80 = int(np.argmax(cum >= 0.8)) + 1
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.arange(1, len(cum) + 1), cum, marker="o")
ax.axhline(0.8, ls="--", color="grey")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("number of origins (ranked by volume)")
ax.set_ylabel("cumulative share of records")
ax.set_title(f"Catchment concentration: {n80} of {len(cum)} origins supply 80% of traffic")
show()

In [ ]:
fig = px.treemap(org_vol, path=["origin"], values="count", color="count",
                 color_continuous_scale="Oranges", title="Origin share of Waterfront traffic")
fig.update_traces(texttemplate="%{label}<br>%{percentRoot:.1%}")
fig.show()

## 4. Arrival patterns over time

In [ ]:
arr = df.groupBy("t").count().toPandas().set_index("t")["count"]
arr = arr.reindex(pd.date_range(arr.index.min(), arr.index.max(), freq=f"{RES_MIN}min"), fill_value=0)
arr.index.name = "t"
smooth_w = max(3, round(30 / RES_MIN))

fig = go.Figure()
fig.add_trace(go.Scatter(x=arr.index, y=arr.values, mode="lines", name=f"arrivals / {RES_MIN} min", opacity=0.55))
fig.add_trace(go.Scatter(x=arr.index, y=arr.rolling(smooth_w, center=True, min_periods=1).mean(),
                         mode="lines", name="smoothed", line=dict(width=3)))
fig.update_layout(title=f"Arrivals per {RES_MIN} minutes", height=450, yaxis_title="arrivals")
fig.update_xaxes(rangeslider_visible=True)
fig.show()

In [ ]:
tb = df.groupBy("tbin").count().orderBy("tbin").toPandas()
if MULTI_DAY:
    tb["count"] = tb["count"] / N_DAYS
fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(tb["tbin"], tb["count"], color="steelblue")
ax.set_xlabel(TBIN_LABEL)
ax.set_ylabel("avg records per day" if MULTI_DAY else "records")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_k))
ax.set_title(f"Arrivals by {TBIN_LABEL}")
plt.xticks(rotation=90 if len(tb) > 24 else 0)
show()

In [ ]:
# Cumulative arrivals: when had 25 / 50 / 75 / 90% of all visitors arrived?
c = arr.cumsum() / arr.sum()
marks = {q: c.index[int(np.argmax(c.values >= q))] for q in [0.25, 0.5, 0.75, 0.9]}
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(c.index, c.values, lw=2.5)
for q, t in marks.items():
    ax.axvline(t, ls=":", color="grey")
    ax.text(t, q, f" {q:.0%} by {t:%H:%M}", fontsize=9, va="bottom")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Cumulative share of arrivals")
show()

In [ ]:
# Peak periods table
peaks = arr.sort_values(ascending=False).head(15).rename("arrivals").reset_index()
peaks["vs_median"] = peaks["arrivals"] / arr.median()
display(peaks)

In [ ]:
# Multi-day views (only shown if the data spans 3+ days)
if MULTI_DAY:
    daily = df.groupBy("date").count().orderBy("date").toPandas()
    daily["date"] = pd.to_datetime(daily["date"])
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(daily["date"], daily["count"], marker="o")
    for d in daily.loc[daily["date"].dt.dayofweek >= 5, "date"]:
        ax.axvspan(d, d + pd.Timedelta(days=1), color="grey", alpha=0.12)
    ax.set_title("Daily records (grey = weekend)")
    show()

    dh = (df.groupBy("date", "dow", "dow_name", "hour").count()
            .groupBy("dow", "dow_name", "hour").agg(F.avg("count").alias("avg")).toPandas())
    piv = dh.pivot_table(index=["dow", "dow_name"], columns="hour", values="avg").sort_index()
    piv.index = piv.index.get_level_values(1)
    plt.figure(figsize=(16, 4.5))
    sns.heatmap(piv, cmap="rocket_r", cbar_kws={"label": "avg records"})
    plt.title("Average records: day of week × hour")
    show()
else:
    print(f"Data covers {N_DAYS} day(s), so the day-of-week / weekday-vs-weekend analysis is skipped.")

## 5. Origin × time: when does each neighbourhood arrive?

In [ ]:
dfo = df.withColumn("origin_grp", F.when(F.col("origin").isin(TOP_ORIGINS), F.col("origin")).otherwise(F.lit("Other")))
ORDER = TOP_ORIGINS + (["Other"] if len(org_vol) > TOP_N else [])

ot = dfo.groupBy("origin_grp", "t").count().toPandas()
ot_piv = (ot.pivot(index="t", columns="origin_grp", values="count")
            .reindex(arr.index).fillna(0)[ORDER])

fig = go.Figure()
for o in ORDER:
    fig.add_trace(go.Scatter(x=ot_piv.index, y=ot_piv[o], stackgroup="one", name=o, mode="lines", line=dict(width=0.5)))
fig.update_layout(title=f"Arrivals per {RES_MIN} min by origin (stacked)", height=500)
fig.show()

share = ot_piv.div(ot_piv.sum(axis=1).replace(0, np.nan), axis=0)
fig = go.Figure()
for o in ORDER:
    fig.add_trace(go.Scatter(x=share.index, y=share[o], stackgroup="one", groupnorm="percent", name=o, mode="lines", line=dict(width=0.5)))
fig.update_layout(title="Origin mix over time (100% stacked)", height=500, yaxis_title="% of arrivals")
fig.show()

In [ ]:
oh = df.groupBy("origin", "tbin").count().toPandas()
oh_piv = oh.pivot(index="origin", columns="tbin", values="count").fillna(0)
oh_piv = oh_piv.loc[org_vol["origin"].head(40)]
oh_share = oh_piv.div(oh_piv.sum(axis=1), axis=0)

plt.figure(figsize=(16, max(5, 0.4 * len(oh_share))))
sns.heatmap(oh_share, cmap="mako_r", cbar_kws={"label": "share of origin's arrivals"})
plt.title(f"When does each origin arrive? (row-normalised, {TBIN_LABEL})")
plt.xlabel(TBIN_LABEL)
show()

In [ ]:
# Arrival window per origin: p10 / median / p90 arrival time
win = (df.groupBy("origin")
         .agg(F.count("*").alias("records"),
              F.expr("timestamp_seconds(percentile_approx(unix_timestamp(ts), 0.1))").alias("p10"),
              F.expr("timestamp_seconds(percentile_approx(unix_timestamp(ts), 0.5))").alias("p50"),
              F.expr("timestamp_seconds(percentile_approx(unix_timestamp(ts), 0.9))").alias("p90"))
         .toPandas().sort_values("p50"))

fig = go.Figure()
fig.add_trace(go.Scatter(x=win["p90"], y=win["origin"], mode="markers", marker=dict(symbol="line-ns-open", size=12), name="p90"))
fig.add_trace(go.Scatter(x=win["p10"], y=win["origin"], mode="markers", marker=dict(symbol="line-ns-open", size=12), name="p10"))
for _, r in win.iterrows():
    fig.add_shape(type="line", x0=r["p10"], x1=r["p90"], y0=r["origin"], y1=r["origin"], line=dict(color="lightgrey", width=6))
fig.add_trace(go.Scatter(x=win["p50"], y=win["origin"], mode="markers",
                         marker=dict(size=8 + 22 * win["records"] / win["records"].max(), color="darkorange"),
                         name="median (size = volume)"))
fig.update_layout(title="Arrival window by origin (p10 – median – p90)", height=max(450, 22 * len(win)))
fig.show()

## 6. Dwell time
Check `DWELL_UNIT_SECONDS` first. Short dwell = commuters passing through; long dwell = people waiting or lingering (crowding and security relevance).

In [ ]:
qs = [0.01, 0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
qv = df.approxQuantile("dwell_time", qs, 0.001)
dwell_q = dict(zip(qs, qv))
display(pd.DataFrame({"quantile": qs, "dwell_time": qv}))
P99, CAP = dwell_q[0.99], dwell_q[0.999]

In [ ]:
BIN = max(1, round(CAP / 80))
hist = (df.filter(F.col("dwell_time") <= CAP)
          .withColumn("bin", F.floor(F.col("dwell_time") / BIN) * BIN)
          .groupBy("bin").count().orderBy("bin").toPandas())

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for a, log in zip(axes, [False, True]):
    a.bar(hist["bin"], hist["count"], width=BIN * 0.9, color="teal")
    a.axvline(dwell_q[0.5], color="red", ls="--", label=f"median {dwell_q[0.5]:.0f}")
    a.axvline(P99, color="black", ls=":", label=f"p99 {P99:.0f}")
    a.set_xlabel("dwell_time")
    a.legend()
    if log:
        a.set_yscale("log")
axes[0].set_title(f"Dwell-time histogram (≤ p99.9, bin = {BIN})")
axes[1].set_title("Same, log scale (tail shape)")
show()

grid = np.linspace(0, 1, 201)
ecdf = df.approxQuantile("dwell_time", grid.tolist(), 0.001)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(ecdf, grid)
ax.set_xscale("log")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("dwell_time (log)")
ax.set_title("Dwell-time ECDF")
show()

In [ ]:
# Dwell by origin: box plots from exact quantiles over all rows
bq = (df.groupBy("origin")
        .agg(F.expr("percentile_approx(dwell_time, array(0.05, 0.25, 0.5, 0.75, 0.95), 1000)").alias("q"),
             F.avg("dwell_time").alias("mean"), F.count("*").alias("records"))
        .toPandas())
bq["median"] = bq["q"].apply(lambda q: q[2])
bq = bq.sort_values("median")

fig = go.Figure()
for _, r in bq.iterrows():
    fig.add_trace(go.Box(y=[r["origin"]], lowerfence=[r["q"][0]], q1=[r["q"][1]], median=[r["q"][2]],
                         q3=[r["q"][3]], upperfence=[r["q"][4]], mean=[r["mean"]], orientation="h",
                         name=r["origin"], showlegend=False))
fig.update_layout(title="Dwell time by origin (whiskers = p5–p95, dashed = mean)",
                  xaxis_title="dwell_time", height=max(450, 24 * len(bq)))
fig.show()

In [ ]:
# Volume vs dwell per origin: many riders + long dwell = biggest contributors to crowding
fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(bq["records"], bq["median"], s=120, alpha=0.75, edgecolor="k", color="darkorange")
for _, r in bq.iterrows():
    ax.annotate(r["origin"], (r["records"], r["median"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.axvline(bq["records"].median(), ls="--", color="grey")
ax.axhline(bq["median"].median(), ls="--", color="grey")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(fmt_k))
ax.set_xlabel("records")
ax.set_ylabel("median dwell")
ax.set_title("Origins: volume vs dwell (top-right = biggest crowding contributors)")
show()

In [ ]:
seg = F.when(F.col("dwell_time") < DWELL_BINS[1], DWELL_LABELS[0])
for hi, lab in zip(DWELL_BINS[2:-1], DWELL_LABELS[1:-1]):
    seg = seg.when(F.col("dwell_time") < hi, lab)
seg = seg.otherwise(DWELL_LABELS[-1])
df = df.withColumn("dwell_segment", seg)

segs = (df.groupBy("origin", "dwell_segment").count().toPandas()
          .pivot(index="origin", columns="dwell_segment", values="count").fillna(0)
          .reindex(columns=DWELL_LABELS, fill_value=0))
segs = segs.div(segs.sum(axis=1), axis=0).sort_values(DWELL_LABELS[0])
ax = segs.plot(kind="barh", stacked=True, figsize=(13, max(5, 0.35 * len(segs))), colormap="viridis")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Dwell segments by origin")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
show()

In [ ]:
# Dwell by arrival time
hd = (df.groupBy("tbin")
        .agg(F.expr("percentile_approx(dwell_time, array(0.25, 0.5, 0.75, 0.9), 1000)").alias("q"))
        .orderBy("tbin").toPandas())
q = np.vstack(hd["q"].values)
fig, ax = plt.subplots(figsize=(15, 5))
x = np.arange(len(hd))
ax.fill_between(x, q[:, 0], q[:, 2], alpha=0.25, label="IQR")
ax.plot(x, q[:, 1], marker="o", label="median")
ax.plot(x, q[:, 3], ls="--", label="p90")
ax.set_xticks(x)
ax.set_xticklabels(hd["tbin"], rotation=90 if len(hd) > 24 else 0)
ax.set_xlabel(TBIN_LABEL)
ax.set_title("Dwell time by arrival time")
ax.legend()
show()

odh = (df.groupBy("origin", "tbin").agg(F.expr("percentile_approx(dwell_time, 0.5)").alias("median_dwell"))
         .toPandas().pivot(index="origin", columns="tbin", values="median_dwell"))
odh = odh.loc[[o for o in org_vol["origin"].head(40) if o in odh.index]]
plt.figure(figsize=(16, max(5, 0.4 * len(odh))))
sns.heatmap(odh, cmap="YlGnBu", cbar_kws={"label": "median dwell"})
plt.title(f"Median dwell: origin × {TBIN_LABEL}")
show()

In [ ]:
frac = min(1.0, SAMPLE_ROWS / N)
sm = (df.sample(fraction=frac, seed=42)
        .select(F.unix_timestamp("ts").alias("u"), "dwell_time")
        .filter(F.col("dwell_time") <= CAP).toPandas())
sm["minutes_from_start"] = (sm["u"] - sm["u"].min()) / 60
fig, ax = plt.subplots(figsize=(13, 5))
hb = ax.hexbin(sm["minutes_from_start"], sm["dwell_time"], gridsize=70, cmap="viridis", bins="log", mincnt=1)
plt.colorbar(hb, label="log count")
ax.set_xlabel(f"minutes since {rng['first']:%Y-%m-%d %H:%M}")
ax.set_ylabel("dwell_time")
ax.set_title(f"Arrival time vs dwell (sample of {len(sm):,})")
show()

## 7. Occupancy: how many devices are in the station at once?
Each record is present from `timestamp` to `timestamp + dwell_time`. Occupancy = cumulative arrivals − cumulative departures.
*Caveat:* devices that were already present before the first record aren't in the data, so the first ~median-dwell of the window is under-counted (warm-up).

In [ ]:
dep = df.groupBy(F.col("t_end").alias("t")).count().toPandas().set_index("t")["count"]
idx = pd.date_range(min(arr.index.min(), dep.index.min()), max(arr.index.max(), dep.index.max()), freq=f"{RES_MIN}min")
flow = pd.DataFrame({"arrivals": arr.reindex(idx, fill_value=0), "departures": dep.reindex(idx, fill_value=0)})
flow.index.name = "t"
flow["net"] = flow["arrivals"] - flow["departures"]
flow["occupancy"] = flow["net"].cumsum()

pk_t = flow["occupancy"].idxmax()
print(f"Peak occupancy: {flow['occupancy'].max():,.0f} devices at {pk_t}")
print(f"Median occupancy while station active: {flow.loc[flow['occupancy'] > 0, 'occupancy'].median():,.0f}")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=flow.index, y=flow["arrivals"], name="arrivals", marker_color="#4C78A8", opacity=0.6), secondary_y=False)
fig.add_trace(go.Bar(x=flow.index, y=-flow["departures"], name="departures (−)", marker_color="#E45756", opacity=0.6), secondary_y=False)
fig.add_trace(go.Scatter(x=flow.index, y=flow["occupancy"], name="occupancy", line=dict(color="black", width=3)), secondary_y=True)
for cap in CAPACITY_SCENARIOS:
    fig.add_trace(go.Scatter(x=[flow.index.min(), flow.index.max()], y=[cap, cap], mode="lines", name=f"capacity {cap:,}",
                             line=dict(color="red", dash="dash")), secondary_y=True)
fig.update_layout(title=f"Station flow per {RES_MIN} min and occupancy", barmode="relative", height=520)
fig.update_yaxes(title_text="arrivals / departures", secondary_y=False)
fig.update_yaxes(title_text="devices present", secondary_y=True)
fig.update_xaxes(rangeslider_visible=True)
fig.show()

for cap in CAPACITY_SCENARIOS:
    above = flow[flow["occupancy"] > cap]
    print(f"Capacity {cap:,}: exceeded for {len(above) * RES_MIN} min"
          + (f" (first {above.index.min()}, last {above.index.max()})" if len(above) else ""))

In [ ]:
# Occupancy by origin: who makes up the crowd?
ao = dfo.groupBy("origin_grp", "t").count().toPandas().pivot(index="t", columns="origin_grp", values="count")
do = dfo.groupBy("origin_grp", F.col("t_end").alias("t")).count().toPandas().pivot(index="t", columns="origin_grp", values="count")
occ_o = ao.reindex(idx).fillna(0).sub(do.reindex(idx).fillna(0), fill_value=0).cumsum().reindex(columns=ORDER, fill_value=0)

fig = go.Figure()
for o in ORDER:
    fig.add_trace(go.Scatter(x=occ_o.index, y=occ_o[o], stackgroup="one", name=o, mode="lines", line=dict(width=0.5)))
fig.update_layout(title="Devices present by origin (stacked)", height=500)
fig.show()

at_peak = occ_o.loc[pk_t].sort_values()
fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(at_peak))))
ax.barh(at_peak.index, at_peak.values, color="darkorange")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(fmt_k))
ax.set_title(f"Crowd composition at peak occupancy ({pk_t:%H:%M})")
show()

In [ ]:
# Which dwell segments make up the crowd over time?
sa = df.groupBy("dwell_segment", "t").count().toPandas().pivot(index="t", columns="dwell_segment", values="count")
sd = df.groupBy("dwell_segment", F.col("t_end").alias("t")).count().toPandas().pivot(index="t", columns="dwell_segment", values="count")
occ_s = sa.reindex(idx).fillna(0).sub(sd.reindex(idx).fillna(0), fill_value=0).cumsum().reindex(columns=DWELL_LABELS, fill_value=0)
fig = go.Figure()
for s in DWELL_LABELS:
    fig.add_trace(go.Scatter(x=occ_s.index, y=occ_s[s], stackgroup="one", name=s, mode="lines", line=dict(width=0.5)))
fig.update_layout(title="Devices present by dwell segment (stacked)", height=480)
fig.show()

In [ ]:
if MULTI_DAY:
    fl = flow.copy()
    fl["date"], fl["hour"] = fl.index.date, fl.index.hour
    hm = fl.groupby(["date", "hour"])["occupancy"].max().unstack()
    plt.figure(figsize=(16, max(4, 0.3 * len(hm))))
    sns.heatmap(hm, cmap="rocket_r", cbar_kws={"label": "max devices present"})
    plt.title("Peak occupancy: date × hour")
    show()

## 8. Security signals
Everything here is aggregate and anonymous: counts and distributions, no individual tracking.
* **8a Surges**: arrivals far above the recent baseline (robust z-score vs trailing median/MAD)
* **8b Origin surges**: a single neighbourhood suddenly over-represented
* **8c Crowd-mix shift**: the origin composition departs from normal (Jensen–Shannon divergence)
* **8d Rapid accumulation**: occupancy rising fast
* **8e Long dwell / loitering**: records above the p99 dwell

In [ ]:
W = max(5, round(60 / RES_MIN))   # ~1-hour trailing baseline

def robust_z(s, w=W):
    past = s.shift(1).rolling(w, min_periods=max(3, w // 2))
    med = past.median()
    mad = past.apply(lambda x: np.median(np.abs(x - np.median(x))), raw=True)
    return (s - med) / (1.4826 * mad).replace(0, np.nan), med

flow["z"], flow["baseline"] = robust_z(flow["arrivals"])
flow["surge"] = flow["z"] > Z_THRESH
print(f"8a: surge buckets (z > {Z_THRESH}): {int(flow['surge'].sum())} of {flow['z'].notna().sum()}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=flow.index, y=flow["arrivals"], mode="lines", name="arrivals"))
fig.add_trace(go.Scatter(x=flow.index, y=flow["baseline"], mode="lines", name="trailing baseline", line=dict(dash="dash")))
s = flow[flow["surge"]]
fig.add_trace(go.Scatter(x=s.index, y=s["arrivals"], mode="markers", name="surge", marker=dict(color="red", size=9)))
fig.update_layout(title="8a. Arrival surges vs trailing 1-hour baseline", height=450)
fig.update_xaxes(rangeslider_visible=True)
fig.show()
display(flow[flow["surge"]].sort_values("z", ascending=False).head(20).reset_index())

In [ ]:
# 8b. Per-origin surges
zo = pd.DataFrame({o: robust_z(ot_piv[o])[0] for o in ORDER})
plt.figure(figsize=(16, max(4, 0.45 * len(ORDER))))
sns.heatmap(zo.T.clip(-5, 5), cmap="vlag", center=0, cbar_kws={"label": "robust z (clipped ±5)"},
            xticklabels=max(1, len(zo) // 24))
plt.title("8b. Arrival surges by origin")
show()

cnt = (zo > Z_THRESH).sum().sort_values()
fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(cnt))))
ax.barh(cnt.index, cnt.values, color="crimson")
ax.set_title(f"8b. Surge buckets per origin (z > {Z_THRESH})")
show()

In [ ]:
# 8c. Crowd-mix shift: JSD between each bucket's origin mix and the overall mix
MIN_ARR = 30
P = ot_piv[ORDER]
p = P.div(P.sum(axis=1).replace(0, np.nan), axis=0)
qv_ = P.sum() / P.values.sum()
m = 0.5 * (p + qv_)
with np.errstate(divide="ignore", invalid="ignore"):
    kl_pm = (p * np.log(p / m)).fillna(0).sum(axis=1)
    kl_qm = (qv_ * np.log(qv_ / m)).fillna(0).sum(axis=1)
flow["mix_jsd"] = (0.5 * kl_pm + 0.5 * kl_qm).where(P.sum(axis=1) >= MIN_ARR).reindex(flow.index)
thr = flow["mix_jsd"].quantile(0.95)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=flow.index, y=flow["arrivals"], name="arrivals", line=dict(color="lightgrey")), secondary_y=False)
fig.add_trace(go.Scatter(x=flow.index, y=flow["mix_jsd"], name="origin-mix divergence", line=dict(color="purple")), secondary_y=True)
fig.add_trace(go.Scatter(x=[flow.index.min(), flow.index.max()], y=[thr, thr], mode="lines", name="p95",
                         line=dict(color="purple", dash="dash")), secondary_y=True)
fig.update_layout(title="8c. Unusual crowd composition (origin mix vs normal)", height=450)
fig.show()
display(flow.sort_values("mix_jsd", ascending=False).head(10)[["arrivals", "mix_jsd"]].reset_index())

In [ ]:
# 8d. Rapid accumulation: occupancy change over the last ~15 minutes
k = max(1, round(15 / RES_MIN))
flow["occ_change_15m"] = flow["occupancy"].diff(k)
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(flow.index, flow["occ_change_15m"], color="darkred")
ax.axhline(0, color="grey", lw=0.8)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_k))
ax.set_title("8d. Change in devices present over ~15 minutes (+ = crowd building)")
show()
display(flow.sort_values("occ_change_15m", ascending=False).head(10)[["arrivals", "departures", "occupancy", "occ_change_15m"]].reset_index())

In [ ]:
# 8e. Long dwell (> p99): potential loitering or people stuck waiting
long_df = df.filter(F.col("dwell_time") > P99)
la = long_df.groupBy("t").count().toPandas().set_index("t")["count"].reindex(idx, fill_value=0)
ld = long_df.groupBy(F.col("t_end").alias("t")).count().toPandas().set_index("t")["count"].reindex(idx, fill_value=0)
flow["long_dwell_present"] = (la - ld).cumsum()

fig, axes = plt.subplots(1, 2, figsize=(20, 5))
axes[0].plot(flow.index, flow["long_dwell_present"], color="darkred")
axes[0].set_title(f"Long-dwell devices (> {P99:.0f}) present")
ls = (df.groupBy("origin").agg(F.avg((F.col("dwell_time") > P99).cast("int")).alias("share"))
        .toPandas().sort_values("share"))
axes[1].barh(ls["origin"], ls["share"], color="darkred")
axes[1].axvline(0.01, ls="--", color="grey", label="1% (expected if random)")
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title("Share of long-dwell records by origin")
axes[1].legend()
show()

night_n = df.filter(F.col("hour") < 5).count()
if MULTI_DAY and night_n:
    ns = (df.groupBy("origin").agg(F.avg((F.col("hour") < 5).cast("int")).alias("night_share"))
            .toPandas().sort_values("night_share"))
    fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(ns))))
    ax.barh(ns["origin"], ns["night_share"], color="midnightblue")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title("Share of records 00:00–04:59 by origin")
    show()

## 9. Relationships

In [ ]:
# Does crowding change behaviour? arrivals / occupancy vs median dwell of people arriving in that bucket
md_t = (df.groupBy("t").agg(F.expr("percentile_approx(dwell_time, 0.5)").alias("median_dwell"))
          .toPandas().set_index("t")["median_dwell"])
rel = flow.join(md_t, how="inner").dropna(subset=["median_dwell"])
rel = rel[rel["arrivals"] >= 30]

fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
for ax, xcol in zip(axes, ["arrivals", "occupancy"]):
    ax.scatter(rel[xcol], rel["median_dwell"], alpha=0.5, s=18)
    b = np.polyfit(rel[xcol], rel["median_dwell"], 1)
    xs = np.linspace(rel[xcol].min(), rel[xcol].max(), 50)
    ax.plot(xs, np.polyval(b, xs), color="red")
    rho = rel[[xcol, "median_dwell"]].corr(method="spearman").iloc[0, 1]
    ax.set_title(f"{xcol} vs median dwell (Spearman ρ = {rho:.2f})")
    ax.set_xlabel(f"{xcol} per {RES_MIN} min" if xcol == "arrivals" else "devices present")
    ax.set_ylabel("median dwell of arrivals")
show()

In [ ]:
# Origins with similar arrival rhythms (candidate groups for targeted service / messaging)
if len(oh_share) >= 3:
    g = sns.clustermap(oh_share.T.corr(), cmap="vlag", center=0, figsize=(12, 12))
    g.fig.suptitle("Correlation of origin arrival profiles", y=1.02)
    plt.show()

## 10. Catchment map
The data has no origin coordinates, so the lookup below uses **approximate neighbourhood/municipality centroids**. Origins not in the lookup are printed; add them to `ORIGIN_COORDS` if needed.

In [ ]:
ORIGIN_COORDS = {
    # City of Vancouver local areas
    "downtown": (49.2820, -123.1180), "west end": (49.2860, -123.1340), "strathcona": (49.2770, -123.0880),
    "grandview-woodland": (49.2740, -123.0670), "hastings-sunrise": (49.2780, -123.0400),
    "kitsilano": (49.2680, -123.1650), "fairview": (49.2640, -123.1300), "mount pleasant": (49.2630, -123.1000),
    "arbutus ridge": (49.2500, -123.1610), "arbutus-ridge": (49.2500, -123.1610), "shaughnessy": (49.2460, -123.1390),
    "south cambie": (49.2460, -123.1200), "riley park": (49.2440, -123.1000),
    "kensington-cedar cottage": (49.2470, -123.0740), "renfrew-collingwood": (49.2480, -123.0400),
    "west point grey": (49.2680, -123.2000), "dunbar-southlands": (49.2400, -123.1850),
    "kerrisdale": (49.2310, -123.1560), "oakridge": (49.2280, -123.1200), "sunset": (49.2210, -123.0950),
    "victoria-fraserview": (49.2190, -123.0650), "killarney": (49.2180, -123.0390), "marpole": (49.2100, -123.1300),
    "stanley park": (49.3017, -123.1417),
    # Metro Vancouver municipalities
    "west vancouver": (49.3280, -123.1600), "north vancouver": (49.3200, -123.0720),
    "burnaby": (49.2488, -122.9805), "richmond": (49.1666, -123.1336), "surrey": (49.1913, -122.8490),
    "new westminster": (49.2057, -122.9110), "coquitlam": (49.2838, -122.7932),
    "port coquitlam": (49.2625, -122.7811), "port moody": (49.2849, -122.8678), "delta": (49.0847, -123.0587),
    "langley": (49.1044, -122.6600), "maple ridge": (49.2193, -122.5984), "pitt meadows": (49.2210, -122.6890),
    "white rock": (49.0253, -122.8026), "ubc": (49.2606, -123.2460), "university endowment lands": (49.2606, -123.2460),
}

def norm(s):
    return " ".join(str(s).lower().replace("–", "-").split())

dest = df.select("location_name", "latitude", "longitude").first()
cm = org_vol.copy()
cm["lat"] = cm["origin"].map(lambda o: ORIGIN_COORDS.get(norm(o), (None, None))[0])
cm["lon"] = cm["origin"].map(lambda o: ORIGIN_COORDS.get(norm(o), (None, None))[1])
unmatched = cm[cm["lat"].isna()]["origin"].tolist()
if unmatched:
    print("No coordinates for:", unmatched)
cm = cm.dropna(subset=["lat"])

fig = go.Figure()
wmax = cm["count"].max()
for _, r in cm.iterrows():
    fig.add_trace(go.Scattermapbox(lat=[r["lat"], dest["latitude"]], lon=[r["lon"], dest["longitude"]], mode="lines",
                                   line=dict(width=1 + 12 * r["count"] / wmax, color="rgba(245,133,24,0.55)"),
                                   hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scattermapbox(lat=cm["lat"], lon=cm["lon"], mode="markers+text", text=cm["origin"], textposition="top center",
                               marker=dict(size=8 + 30 * cm["count"] / wmax, color="#4C78A8"),
                               customdata=np.stack([cm["count"], cm["share"] * 100], axis=1),
                               hovertemplate="%{text}<br>%{customdata[0]:,} records (%{customdata[1]:.1f}%)<extra></extra>",
                               name="origins"))
fig.add_trace(go.Scattermapbox(lat=[dest["latitude"]], lon=[dest["longitude"]], mode="markers+text",
                               text=[dest["location_name"]], textposition="bottom center",
                               marker=dict(size=22, color="red"), name="station"))
fig.update_layout(mapbox_style="open-street-map", mapbox_zoom=10, height=700,
                  mapbox_center=dict(lat=float(dest["latitude"]) - 0.03, lon=float(dest["longitude"]) + 0.03),
                  title="Where Waterfront's visitors come from (approx. origin centroids; line width = volume)")
fig.show()

## 11. KPI tables
Two Delta tables that feed the solution (dashboard, alerting thresholds, forecasting):
* `eda_origin_kpis`: one row per origin
* `eda_station_timeline`: one row per time bucket (arrivals, departures, occupancy, surge flags)

In [ ]:
peak_bin = oh_piv.idxmax(axis=1).rename("peak_" + ("hour" if MULTI_DAY else "window"))
origin_kpis = (org_vol.rename(columns={"count": "records"})
    .merge(bq[["origin", "mean", "median"]].rename(columns={"mean": "mean_dwell", "median": "median_dwell"}), on="origin")
    .merge(win[["origin", "p10", "p50", "p90"]].rename(columns={"p10": "arrival_p10", "p50": "arrival_median", "p90": "arrival_p90"}), on="origin")
    .merge(ls.rename(columns={"share": "long_dwell_share"}), on="origin")
    .merge(peak_bin.reset_index(), on="origin", how="left"))
origin_kpis["surge_buckets"] = origin_kpis["origin"].map(cnt).fillna(0).astype(int)   # top origins only
origin_kpis["present_at_peak"] = origin_kpis["origin"].map(at_peak)
display(origin_kpis)

timeline = flow.reset_index().rename(columns={"index": "t"})
display(timeline.head(20))

spark.createDataFrame(origin_kpis).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT_SCHEMA}.eda_origin_kpis")
spark.createDataFrame(timeline).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT_SCHEMA}.eda_station_timeline")
print(f"Saved {OUT_SCHEMA}.eda_origin_kpis and {OUT_SCHEMA}.eda_station_timeline")